# Mexico Toy Sales：零售经营与库存配置决策分析

## 01. Business Objective & Data Foundation

本章回答：我们要解决什么业务问题，数据是否足够可靠？在正式分析前，先确认销售、商品、门店和库存数据的粒度及关联关系，并特别识别不能直接解释为零库存的缺失 Store-SKU 记录。

本项目是一个多门店玩具零售经营与库存配置分析项目。本阶段先建立经营监控、变化诊断与商品优先级；后续阶段再结合近期需求和单时点库存，分析库存健康、跨门店错配以及调拨与补货优先级。

### 核心业务问题

1. 整体销售和盈利表现如何？
2. 是否存在值得关注的经营变化？
3. 哪些品类、商品和门店贡献了主要变化？
4. 哪些商品对毛利润最重要？
5. 后续如何结合近期需求与库存判断缺货、积压和库存错配？

**Objective：**提高盈利性销售表现与库存配置效率。

**Strategy：**监控销售和盈利表现；定位经营变化及主要贡献来源；识别高价值商品并建立优先级；后续结合近期需求诊断库存健康，并识别跨门店库存错配、调拨候选与补货优先级。

### Measurement Framework

经营核心指标：`Total Sales`、`Total Units`、`COGS`、`Gross Profit`、`Gross Margin`、`Monthly Sales MoM`。

诊断维度：`Month`、`Category`、`Product`、`Store`、`Store Location / City`。

后续库存指标：`Recent Demand`、`Days of Cover`、`Active Stockout`、`Critical`、`Overstock`、`Dormant`、`GP Opportunity`、`Reallocation Coverage`。

本项目不使用 AARRR、Funnel、Cohort、A/B Test 或 Machine Learning，因为这些方法与当前零售经营和单时点库存配置问题不匹配。

### Data Loading & Understanding

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('data')
sales = pd.read_csv(DATA_DIR / 'sales.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
stores = pd.read_csv(DATA_DIR / 'stores.csv')
inventory = pd.read_csv(DATA_DIR / 'inventory.csv')

In [ ]:
table_specs = {
    'sales': {'df': sales, 'grain': '一条销售记录', 'key': 'Sale_ID'},
    'products': {'df': products, 'grain': '一个商品', 'key': 'Product_ID'},
    'stores': {'df': stores, 'grain': '一家门店', 'key': 'Store_ID'},
    'inventory': {'df': inventory, 'grain': '一个实际记录的 Store-SKU 组合', 'key': 'Store_ID + Product_ID'},
}
table_overview = pd.DataFrame([{'Table':name,'Rows':len(spec['df']),'Columns':spec['df'].shape[1],'Field_Names':', '.join(spec['df'].columns),'Data_Types':', '.join(f'{c}: {t}' for c,t in spec['df'].dtypes.items()),'Grain':spec['grain'],'Primary_or_Candidate_Key':spec['key']} for name,spec in table_specs.items()])
display(table_overview)

### Data Quality & Grain Validation

质量检查覆盖缺失、重复、主键、候选键、日期和数值范围、业务规则、孤立外键、关联粒度及库存组合完整性。

In [ ]:
# 复用原有清洗逻辑。
sales['Date']=pd.to_datetime(sales['Date'],errors='raise'); stores['Store_Open_Date']=pd.to_datetime(stores['Store_Open_Date'],errors='raise')
for column in ['Product_Cost','Product_Price']:
    products[column]=products[column].astype(str).str.replace('$','',regex=False).str.strip().astype(float)
missing_summary=pd.DataFrame({name:spec['df'].isna().sum() for name,spec in table_specs.items()}).fillna(0).astype(int).T
duplicate_summary=pd.DataFrame({'Exact_Duplicates':{name:int(spec['df'].duplicated().sum()) for name,spec in table_specs.items()}})
display(missing_summary.style.set_caption('Missing Values')); display(duplicate_summary.style.set_caption('Exact Duplicate Rows'))

In [ ]:
validation_results=pd.DataFrame([['sales','Sale_ID is unique',sales['Sale_ID'].is_unique],['sales','Units > 0',sales['Units'].gt(0).all()],['sales','No orphan Store_ID',sales['Store_ID'].isin(stores['Store_ID']).all()],['sales','No orphan Product_ID',sales['Product_ID'].isin(products['Product_ID']).all()],['products','Product_ID is unique',products['Product_ID'].is_unique],['products','Product_Price > 0',products['Product_Price'].gt(0).all()],['products','Product_Cost > 0',products['Product_Cost'].gt(0).all()],['products','Product_Price >= Product_Cost',products['Product_Price'].ge(products['Product_Cost']).all()],['stores','Store_ID is unique',stores['Store_ID'].is_unique],['inventory','Store_ID + Product_ID is unique',~inventory.duplicated(['Store_ID','Product_ID']).any()],['inventory','Stock_On_Hand >= 0',inventory['Stock_On_Hand'].ge(0).all()],['inventory','No orphan Store_ID',inventory['Store_ID'].isin(stores['Store_ID']).all()],['inventory','No orphan Product_ID',inventory['Product_ID'].isin(products['Product_ID']).all()]],columns=['Table','Validation','Passed'])
range_summary=pd.DataFrame([['Sales Date',sales['Date'].min(),sales['Date'].max()],['Units',sales['Units'].min(),sales['Units'].max()],['Product Cost',products['Product_Cost'].min(),products['Product_Cost'].max()],['Product Price',products['Product_Price'].min(),products['Product_Price'].max()],['Stock On Hand',inventory['Stock_On_Hand'].min(),inventory['Stock_On_Hand'].max()]],columns=['Field','Minimum','Maximum'])
display(validation_results); display(range_summary); assert validation_results['Passed'].all(),'Data quality validation failed.'

### Inventory Combination Data Quality Check

理论 Store-SKU 组合是门店与商品的笛卡尔积。缺失 inventory 记录不等于 `Stock_On_Hand = 0`，必须结合销售记录识别为数据异常或库存未知。

In [ ]:
all_store_sku=pd.MultiIndex.from_product([stores['Store_ID'],products['Product_ID']],names=['Store_ID','Product_ID']).to_frame(index=False)
missing_inventory_combinations=all_store_sku.merge(inventory[['Store_ID','Product_ID']],on=['Store_ID','Product_ID'],how='left',indicator=True).query("_merge == 'left_only'").drop(columns='_merge')
historical_sales=sales.groupby(['Store_ID','Product_ID'],as_index=False)['Units'].sum().rename(columns={'Units':'Historical_Units'})
max_sales_date=sales['Date'].max(); recent_90d_start=max_sales_date-pd.Timedelta(days=89)
recent_90d_sales=sales.loc[sales['Date'].between(recent_90d_start,max_sales_date)].groupby(['Store_ID','Product_ID'],as_index=False)['Units'].sum().rename(columns={'Units':'Recent_90D_Units'})
inventory_exceptions=missing_inventory_combinations.merge(historical_sales,on=['Store_ID','Product_ID'],how='left').merge(recent_90d_sales,on=['Store_ID','Product_ID'],how='left').merge(products[['Product_ID','Product_Name']],on='Product_ID',how='left')
inventory_exceptions[['Historical_Units','Recent_90D_Units']]=inventory_exceptions[['Historical_Units','Recent_90D_Units']].fillna(0)
inventory_combination_summary=pd.DataFrame({'Metric':['Theoretical Store-SKU combinations','Actual inventory records','Missing inventory combinations','Missing combinations with historical sales','Missing combinations with recent 90D sales'],'Value':[len(all_store_sku),len(inventory),len(missing_inventory_combinations),inventory_exceptions['Historical_Units'].gt(0).sum(),inventory_exceptions['Recent_90D_Units'].gt(0).sum()]})
recent_inventory_exceptions=inventory_exceptions.loc[inventory_exceptions['Recent_90D_Units'].gt(0),['Store_ID','Product_ID','Product_Name','Historical_Units','Recent_90D_Units']].sort_values('Recent_90D_Units',ascending=False)
display(inventory_combination_summary); display(recent_inventory_exceptions)

**正式数据质量结论：**缺失 inventory 记录只能标记为 `Inventory Data Exception / Inventory Unknown`，不能解释为 Stockout。最近90天仍有销量的异常组合需要优先核查库存记录及商品经营状态。

### Data Preparation & Fact Construction

In [ ]:
# 复用已验证的 many-to-one 关联。
sales_data=sales.merge(products,on='Product_ID',how='left',validate='many_to_one').merge(stores,on='Store_ID',how='left',validate='many_to_one')
join_validation=pd.DataFrame({'Metric':['Source sales rows','Joined sales rows','Missing Product_Name','Missing Store_Name'],'Value':[len(sales),len(sales_data),sales_data['Product_Name'].isna().sum(),sales_data['Store_Name'].isna().sum()]}); display(join_validation)
assert len(sales_data)==len(sales),'Join changed the sales fact grain.'; assert sales_data[['Product_Name','Store_Name']].notna().all().all(),'Join produced unmatched dimensions.'
# 分析层使用准确业务名称；当前 output CSV 和 Power BI schema 本阶段不修改。
sales_data['Sales']=sales_data['Units']*sales_data['Product_Price']; sales_data['COGS']=sales_data['Units']*sales_data['Product_Cost']; sales_data['Gross_Profit']=sales_data['Sales']-sales_data['COGS']; sales_data['Month']=sales_data['Date'].dt.to_period('M').dt.to_timestamp()
assert np.allclose(sales_data['Sales']-sales_data['COGS'],sales_data['Gross_Profit'])

## 02. Sales & Profit Performance Diagnosis

本章回答：生意表现怎么样、哪个月变化最明显、变化主要集中在哪里？月度变化与贡献分析是描述性诊断，不将观察到的变化解释为异常事故或因果结果。

In [ ]:
total_units=sales_data['Units'].sum(); total_sales=sales_data['Sales'].sum(); total_cogs=sales_data['COGS'].sum(); total_gross_profit=sales_data['Gross_Profit'].sum(); gross_margin=total_gross_profit/total_sales
kpi_summary=pd.DataFrame({'Metric':['Total Units','Total Sales','Total COGS','Gross Profit','Gross Margin'],'Value':[total_units,total_sales,total_cogs,total_gross_profit,gross_margin]}); display(kpi_summary)
category_summary=sales_data.groupby('Product_Category',as_index=False).agg(Units=('Units','sum'),Sales=('Sales','sum'),COGS=('COGS','sum'),Gross_Profit=('Gross_Profit','sum')); category_summary['Gross_Margin']=category_summary['Gross_Profit']/category_summary['Sales']
product_summary=sales_data.groupby(['Product_ID','Product_Name','Product_Category'],as_index=False).agg(Units=('Units','sum'),Sales=('Sales','sum'),COGS=('COGS','sum'),Gross_Profit=('Gross_Profit','sum')); product_summary['Gross_Margin']=product_summary['Gross_Profit']/product_summary['Sales']
store_summary=sales_data.groupby(['Store_ID','Store_Name','Store_City','Store_Location'],as_index=False).agg(Units=('Units','sum'),Sales=('Sales','sum'),COGS=('COGS','sum'),Gross_Profit=('Gross_Profit','sum')); store_summary['Gross_Margin']=store_summary['Gross_Profit']/store_summary['Sales']
monthly_summary=sales_data.groupby('Month',as_index=False).agg(Units=('Units','sum'),Sales=('Sales','sum'),COGS=('COGS','sum'),Gross_Profit=('Gross_Profit','sum')).sort_values('Month'); monthly_summary['Gross_Margin']=monthly_summary['Gross_Profit']/monthly_summary['Sales']; display(monthly_summary)

In [ ]:
fig,ax1=plt.subplots(figsize=(11,5)); ax1.plot(monthly_summary['Month'],monthly_summary['Sales'],marker='o',color='#2F6690'); ax1.set_ylabel('Sales'); ax1.set_title('Monthly Sales and Gross Profit Trend'); ax2=ax1.twinx(); ax2.plot(monthly_summary['Month'],monthly_summary['Gross_Profit'],marker='o',color='#E07A5F'); ax2.set_ylabel('Gross Profit'); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

### Monthly Change Analysis

月度变化分析用于识别值得进一步解释的变化，不预设一定存在异常。由于数据只有约21个月，本项目采用透明的环比和贡献分析，不使用 Isolation Forest、ARIMA、Prophet 或其他复杂异常检测模型。

In [ ]:
monthly_summary['Sales_MoM']=monthly_summary['Sales'].pct_change(); monthly_summary['GP_MoM']=monthly_summary['Gross_Profit'].pct_change(); monthly_summary['Units_MoM']=monthly_summary['Units'].pct_change()
monthly_change_view=monthly_summary[['Month','Sales','Sales_MoM','Gross_Profit','GP_MoM','Gross_Margin']]; display(monthly_change_view)
change_extremes=pd.DataFrame([['Largest Sales MoM increase',monthly_summary.loc[monthly_summary['Sales_MoM'].idxmax(),'Month'],monthly_summary['Sales_MoM'].max()],['Largest Sales MoM decline',monthly_summary.loc[monthly_summary['Sales_MoM'].idxmin(),'Month'],monthly_summary['Sales_MoM'].min()],['Largest GP MoM increase',monthly_summary.loc[monthly_summary['GP_MoM'].idxmax(),'Month'],monthly_summary['GP_MoM'].max()],['Largest GP MoM decline',monthly_summary.loc[monthly_summary['GP_MoM'].idxmin(),'Month'],monthly_summary['GP_MoM'].min()]],columns=['Change','Month','MoM']); display(change_extremes)
fig,ax=plt.subplots(figsize=(11,4.5)); colors=np.where(monthly_summary['Sales_MoM'].fillna(0)>=0,'#4C956C','#D1495B'); ax.bar(monthly_summary['Month'],monthly_summary['Sales_MoM'],width=20,color=colors); ax.axhline(0,color='#333333',linewidth=.8); ax.set_title('Monthly Sales MoM: descriptive change monitoring'); ax.set_ylabel('Sales MoM'); ax.yaxis.set_major_formatter(lambda x,pos:f'{x:.0%}'); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

### Contribution Diagnosis

选择完整月份中 Sales MoM 最大的负向变化作为诊断对象。贡献分析描述变化集中于哪些维度，不构成因果推断；本项目使用“主要贡献来源”“变化集中于”，不使用因果表述。

In [ ]:
target_month=monthly_summary.loc[monthly_summary['Sales_MoM'].idxmin(),'Month']; previous_month=target_month-pd.offsets.MonthBegin(1)
target_sales=monthly_summary.loc[monthly_summary['Month'].eq(target_month),'Sales'].iloc[0]; previous_sales=monthly_summary.loc[monthly_summary['Month'].eq(previous_month),'Sales'].iloc[0]; total_delta_sales=target_sales-previous_sales
selected_change=pd.DataFrame({'Previous_Month':[previous_month],'Target_Month':[target_month],'Previous_Sales':[previous_sales],'Target_Sales':[target_sales],'Delta_Sales':[total_delta_sales],'Sales_MoM':[target_sales/previous_sales-1]}); display(selected_change)
def build_sales_contribution(data,dimensions):
    q=data.loc[data['Month'].isin([previous_month,target_month])].pivot_table(index=dimensions,columns='Month',values='Sales',aggfunc='sum',fill_value=0).reset_index(); q['Delta_Sales']=q[target_month]-q[previous_month]; q['Contribution_Share_of_Net_Change']=q['Delta_Sales']/total_delta_sales; return q
def show_contributors(label,q,dimensions,n=5):
    cols=dimensions+['Delta_Sales','Contribution_Share_of_Net_Change']; print(f'{label} — Top negative contributors'); display(q.nsmallest(n,'Delta_Sales')[cols]); print(f'{label} — Top positive offsets'); display(q.loc[q['Delta_Sales'].gt(0)].nlargest(n,'Delta_Sales')[cols])
category_contribution=build_sales_contribution(sales_data,['Product_Category']); product_contribution=build_sales_contribution(sales_data,['Product_ID','Product_Name','Product_Category']); store_contribution=build_sales_contribution(sales_data,['Store_ID','Store_Name','Store_City','Store_Location'])
show_contributors('Category',category_contribution,['Product_Category']); show_contributors('Product',product_contribution,['Product_ID','Product_Name','Product_Category']); show_contributors('Store',store_contribution,['Store_ID','Store_Name','Store_City','Store_Location'])

贡献份额以净变化为分母。总变化为负时，负向对象得到正的下降贡献份额，正向 offset 的份额为负，表示其抵消了一部分下降。所有品类都可能同时下降，因此某一维度不一定存在正向 offset。

## 03. Product Prioritization

本章回答：哪些商品值得优先管理？主分类继续使用完整历史 Gross Profit Contribution，因为项目关注盈利性销售和后续库存资源优先级。

章节过渡：商品的重要程度不同，因此后续库存风险不能只按缺货数量排序，需要把 ABC 商品优先级与库存健康状态结合。

In [ ]:
product_abc=product_summary[['Product_ID','Product_Name','Product_Category','Sales','Gross_Profit']].sort_values(['Gross_Profit','Product_ID'],ascending=[False,True]).reset_index(drop=True)
product_abc['GP_Share']=product_abc['Gross_Profit']/product_abc['Gross_Profit'].sum(); product_abc['Cumulative_GP_Share']=product_abc['GP_Share'].cumsum(); product_abc['ABC_Class']=np.select([product_abc['Cumulative_GP_Share'].le(.80),product_abc['Cumulative_GP_Share'].le(.95)],['A','B'],default='C')
abc_summary=product_abc.groupby('ABC_Class',as_index=False).agg(Product_Count=('Product_ID','count'),Gross_Profit=('Gross_Profit','sum')); abc_summary['GP_Share']=abc_summary['Gross_Profit']/product_abc['Gross_Profit'].sum(); display(product_abc); display(abc_summary)
assert product_abc['Product_ID'].is_unique and product_abc['ABC_Class'].isin(['A','B','C']).all()
fig,ax1=plt.subplots(figsize=(12,5)); x=np.arange(len(product_abc)); colors=product_abc['ABC_Class'].map({'A':'#2F6690','B':'#E9C46A','C':'#E07A5F'}); ax1.bar(x,product_abc['Gross_Profit'],color=colors); ax1.set_ylabel('Gross Profit'); ax1.set_xlabel('Products ranked by Gross Profit'); ax1.set_title('Product Gross Profit Pareto and ABC Classification'); ax2=ax1.twinx(); ax2.plot(x,product_abc['Cumulative_GP_Share'],color='#333333',marker='o',markersize=3); ax2.axhline(.80,color='#2F6690',linestyle='--'); ax2.axhline(.95,color='#E07A5F',linestyle='--'); ax2.set_ylabel('Cumulative GP Share'); ax2.set_ylim(0,1.05); fig.tight_layout(); plt.show()

## 04. Inventory Health & Risk Assessment

本章回答：当前库存和近期需求是否匹配，哪些库存问题最值得优先处理？

### A. Recent Demand

Base使用最近90个自然日历史销量估算近期销售速度；它是静态需求代理，不是需求预测。

In [ ]:
DEMAND_WINDOW_DAYS=90
max_sales_date=sales['Date'].max(); demand_window_start=max_sales_date-pd.Timedelta(days=DEMAND_WINDOW_DAYS-1)
recent_90d_demand=(sales.loc[sales['Date'].between(demand_window_start,max_sales_date)].groupby(['Store_ID','Product_ID'],as_index=False)['Units'].sum().rename(columns={'Units':'Recent_90D_Units'}))
recent_90d_demand['Avg_Daily_Demand_90D']=recent_90d_demand['Recent_90D_Units']/DEMAND_WINDOW_DAYS
demand_window_summary=pd.DataFrame({'Metric':['Maximum sales date','90D window start','90D window end','Calendar days','Selling Store-SKU combinations'],'Value':[max_sales_date,demand_window_start,max_sales_date,DEMAND_WINDOW_DAYS,len(recent_90d_demand)]})
display(demand_window_summary)

### B–C. Days of Cover & Inventory Health

库存件数本身不能判断风险，需要结合近期销售速度计算 Days of Cover。7D、14D与60D均为分析情景阈值，不是企业真实安全库存或补货政策。

In [ ]:
inventory_analysis=(inventory.merge(products,on='Product_ID',how='left',validate='many_to_one').merge(stores,on='Store_ID',how='left',validate='many_to_one').merge(product_abc[['Product_ID','ABC_Class']],on='Product_ID',how='left',validate='many_to_one').merge(recent_90d_demand,on=['Store_ID','Product_ID'],how='left',validate='one_to_one'))
inventory_analysis['Recent_90D_Units']=inventory_analysis['Recent_90D_Units'].fillna(0).astype(int)
inventory_analysis['Avg_Daily_Demand_90D']=inventory_analysis['Recent_90D_Units']/DEMAND_WINDOW_DAYS
inventory_analysis['Demand_Status']=np.where(inventory_analysis['Recent_90D_Units'].gt(0),'Recent Demand','No Recent Demand')
inventory_analysis['Inventory_Cost']=inventory_analysis['Stock_On_Hand']*inventory_analysis['Product_Cost']
inventory_analysis['Inventory_Retail_Value']=inventory_analysis['Stock_On_Hand']*inventory_analysis['Product_Price']
inventory_analysis['Unit_GP']=inventory_analysis['Product_Price']-inventory_analysis['Product_Cost']
inventory_analysis['Days_Cover']=np.where(inventory_analysis['Recent_90D_Units'].gt(0),inventory_analysis['Stock_On_Hand']/inventory_analysis['Avg_Daily_Demand_90D'],np.nan)
health_conditions=[(inventory_analysis['Stock_On_Hand'].eq(0)&inventory_analysis['Recent_90D_Units'].gt(0)),(inventory_analysis['Stock_On_Hand'].gt(0)&inventory_analysis['Days_Cover'].gt(0)&inventory_analysis['Days_Cover'].le(7)),(inventory_analysis['Days_Cover'].gt(7)&inventory_analysis['Days_Cover'].le(14)),(inventory_analysis['Days_Cover'].gt(14)&inventory_analysis['Days_Cover'].le(60)),inventory_analysis['Days_Cover'].gt(60),(inventory_analysis['Stock_On_Hand'].gt(0)&inventory_analysis['Recent_90D_Units'].eq(0))]
health_labels=['Active Stockout','Critical','Low Stock','Healthy','Overstock','Dormant / No Recent Sales']
inventory_analysis['Inventory_Status']=np.select(health_conditions,health_labels,default='Unclassified')
assert len(inventory_analysis)==len(inventory); assert inventory_analysis['Inventory_Status'].ne('Unclassified').all(); assert inventory_analysis[['Store_ID','Product_ID']].duplicated().sum()==0; assert np.isinf(inventory_analysis['Days_Cover'].fillna(0)).sum()==0

In [ ]:
inventory_health_summary=(inventory_analysis.groupby('Inventory_Status',as_index=False).agg(Store_SKU_Count=('Product_ID','size'),Inventory_Units=('Stock_On_Hand','sum'),Inventory_Cost=('Inventory_Cost','sum'),Product_Count=('Product_ID','nunique'),Store_Count=('Store_ID','nunique')))
display(inventory_health_summary.sort_values('Store_SKU_Count',ascending=False))
fig,ax=plt.subplots(figsize=(10,4.5)); health_plot=inventory_health_summary.sort_values('Store_SKU_Count'); ax.barh(health_plot['Inventory_Status'],health_plot['Store_SKU_Count'],color='#2F6690'); ax.set_title('Inventory Health Distribution — Base 90D Demand'); ax.set_xlabel('Store-SKU Count'); fig.tight_layout(); plt.show()

### Inventory Unknown remains outside Inventory Health

157个缺失 inventory 组合不进入上述六类健康状态。它们仍定义为 `Inventory Data Exception / Inventory Unknown`；其中最近90天有销量的组合属于 Data Quality Investigation Priority，而不是 Stockout、Receiver或Replenishment Priority。

In [ ]:
data_quality_investigation_priority=inventory_exceptions.loc[inventory_exceptions['Recent_90D_Units'].gt(0),['Store_ID','Product_ID','Product_Name','Historical_Units','Recent_90D_Units']].copy()
data_quality_investigation_priority['Exception_Label']='Inventory Data Exception / Inventory Unknown'

### D. ABC × Inventory Risk

Analytical Priority Framework把商品价值与库存状态结合，用于确定先检查哪些Store-SKU，不代表企业已实施的优先级政策。

In [ ]:
RECEIVER_TARGET_DAYS=14
inventory_analysis['Analytical_Need_14D']=np.maximum(np.ceil(inventory_analysis['Avg_Daily_Demand_90D']*RECEIVER_TARGET_DAYS-inventory_analysis['Stock_On_Hand']),0).astype(int)
inventory_analysis['Analytical_GP_Opportunity']=inventory_analysis['Analytical_Need_14D']*inventory_analysis['Unit_GP']
priority_conditions=[inventory_analysis['ABC_Class'].eq('A')&inventory_analysis['Inventory_Status'].isin(['Active Stockout','Critical']),inventory_analysis['ABC_Class'].eq('B')&inventory_analysis['Inventory_Status'].isin(['Active Stockout','Critical'])|inventory_analysis['ABC_Class'].eq('A')&inventory_analysis['Inventory_Status'].eq('Low Stock'),inventory_analysis['Inventory_Status'].isin(['Active Stockout','Critical','Low Stock']),inventory_analysis['ABC_Class'].eq('C')&inventory_analysis['Inventory_Status'].isin(['Overstock','Dormant / No Recent Sales'])]
inventory_analysis['Analytical_Priority']=np.select(priority_conditions,['P1','P2','P3','Inventory Reduction Candidate'],default='Monitor')
priority_summary=inventory_analysis.groupby('Analytical_Priority',as_index=False).agg(Store_SKU_Count=('Product_ID','size'),Need_Units=('Analytical_Need_14D','sum'),GP_Opportunity=('Analytical_GP_Opportunity','sum'),Inventory_Units=('Stock_On_Hand','sum'),Inventory_Cost=('Inventory_Cost','sum'))
abc_status_matrix=pd.crosstab(inventory_analysis['ABC_Class'],inventory_analysis['Inventory_Status'])
display(priority_summary.sort_values('Analytical_Priority')); display(abc_status_matrix)

### E. Opportunity Exposure

Need Qty、Sales Opportunity Exposure与GP Opportunity Exposure描述达到14天目标覆盖的情景规模，不是Lost Sales、Lost Profit或实际损失。

In [ ]:
receiver_pool=inventory_analysis.loc[inventory_analysis['Inventory_Status'].isin(['Active Stockout','Critical'])].copy()
receiver_pool['Receiver_Priority']=np.where(receiver_pool['Inventory_Status'].eq('Active Stockout'),'P1 Active Stockout','P2 Critical')
receiver_pool['Need_Qty']=np.maximum(np.ceil(receiver_pool['Avg_Daily_Demand_90D']*RECEIVER_TARGET_DAYS-receiver_pool['Stock_On_Hand']),0).astype(int)
receiver_pool['Sales_Opportunity_Exposure']=receiver_pool['Need_Qty']*receiver_pool['Product_Price']
receiver_pool['GP_Opportunity_Exposure']=receiver_pool['Need_Qty']*receiver_pool['Unit_GP']
receiver_summary=receiver_pool.groupby('Receiver_Priority',as_index=False).agg(Receiver_Count=('Product_ID','size'),Product_Count=('Product_ID','nunique'),Store_Count=('Store_ID','nunique'),Need_Units=('Need_Qty','sum'),Sales_Opportunity_Exposure=('Sales_Opportunity_Exposure','sum'),GP_Opportunity_Exposure=('GP_Opportunity_Exposure','sum'))
a_class_receiver_summary=pd.DataFrame({'Metric':['A-class Receiver Count','A-class Need Units','A-class GP Opportunity Exposure'],'Value':[receiver_pool['ABC_Class'].eq('A').sum(),receiver_pool.loc[receiver_pool['ABC_Class'].eq('A'),'Need_Qty'].sum(),receiver_pool.loc[receiver_pool['ABC_Class'].eq('A'),'GP_Opportunity_Exposure'].sum()]})
receiver_overview=pd.DataFrame({'Metric':['Active Stockout Receiver','Critical Receiver','Total Receiver Count','Total Need Units','Total Sales Opportunity Exposure','Total GP Opportunity Exposure','A-class Receiver Count','A-class Need Units','A-class GP Opportunity Exposure'],'Value':[receiver_pool['Inventory_Status'].eq('Active Stockout').sum(),receiver_pool['Inventory_Status'].eq('Critical').sum(),len(receiver_pool),receiver_pool['Need_Qty'].sum(),receiver_pool['Sales_Opportunity_Exposure'].sum(),receiver_pool['GP_Opportunity_Exposure'].sum(),receiver_pool['ABC_Class'].eq('A').sum(),receiver_pool.loc[receiver_pool['ABC_Class'].eq('A'),'Need_Qty'].sum(),receiver_pool.loc[receiver_pool['ABC_Class'].eq('A'),'GP_Opportunity_Exposure'].sum()]}); display(receiver_overview)

## 05. Inventory Allocation Decision Support

本章回答：风险是公司整体没货，还是库存放错了门店；现有库存能缓解多少短缺；剩余缺口应如何处理？

### A. Same-SKU Cross-Store Imbalance

同一SKU可同时在部分门店紧缺、在其他门店存在Excess。但 `Donor Availability != Quantity Coverage`：所有Receiver SKU存在Donor，不代表所有Need都能解决。

In [ ]:
BASE_DONOR_RESERVE_DAYS=45
donor_pool=inventory_analysis.loc[inventory_analysis['Recent_90D_Units'].gt(0)&inventory_analysis['Days_Cover'].gt(BASE_DONOR_RESERVE_DAYS)].copy()
donor_pool['Excess_Qty']=np.maximum(np.floor(donor_pool['Stock_On_Hand']-donor_pool['Avg_Daily_Demand_90D']*BASE_DONOR_RESERVE_DAYS),0).astype(int)
donor_pool=donor_pool.loc[donor_pool['Excess_Qty'].gt(0)].copy()
potential_dormant_donor_pool=inventory_analysis.loc[inventory_analysis['Inventory_Status'].eq('Dormant / No Recent Sales')].copy()
donor_summary=pd.DataFrame({'Metric':['Effective Donor Store-SKU','Donor Products','Donor Stores','Total Excess Units','Potential Dormant Donor Store-SKU','Potential Dormant Units','Potential Dormant Inventory Cost'],'Value':[len(donor_pool),donor_pool['Product_ID'].nunique(),donor_pool['Store_ID'].nunique(),donor_pool['Excess_Qty'].sum(),len(potential_dormant_donor_pool),potential_dormant_donor_pool['Stock_On_Hand'].sum(),potential_dormant_donor_pool['Inventory_Cost'].sum()]})
display(donor_summary)

### B. Base Reallocation Scenario

Base固定90D Demand、Receiver Target 14D与Donor Reserve 45D，先在SKU层面对比Receiver Need和Donor Excess，验证理论Maximum Matched Qty与数量覆盖。

In [ ]:
receiver_by_sku=receiver_pool.groupby(['Product_ID','Product_Name'],as_index=False).agg(Total_Receiver_Need=('Need_Qty','sum'),Product_Price=('Product_Price','first'),Unit_GP=('Unit_GP','first'))
donor_by_sku=donor_pool.groupby('Product_ID',as_index=False)['Excess_Qty'].sum().rename(columns={'Excess_Qty':'Total_Donor_Excess'})
sku_imbalance=receiver_by_sku.merge(donor_by_sku,on='Product_ID',how='left'); sku_imbalance['Total_Donor_Excess']=sku_imbalance['Total_Donor_Excess'].fillna(0).astype(int); sku_imbalance['Max_Matched_Qty_SKU']=np.minimum(sku_imbalance['Total_Receiver_Need'],sku_imbalance['Total_Donor_Excess']).astype(int); sku_imbalance['Quantity_Coverage']=sku_imbalance['Max_Matched_Qty_SKU']/sku_imbalance['Total_Receiver_Need']; sku_imbalance['Covered_GP_Opportunity']=sku_imbalance['Max_Matched_Qty_SKU']*sku_imbalance['Unit_GP']
donor_availability_summary=pd.DataFrame({'Metric':['Receiver SKU Count','Receiver SKU with Donor','Receiver rows whose SKU has Donor','Total Receiver Need','Donor Excess for Receiver SKUs','SKU-level Maximum Matched Units','Unit Need Coverage','Covered GP Opportunity','GP Opportunity Coverage'],'Value':[len(sku_imbalance),sku_imbalance['Total_Donor_Excess'].gt(0).sum(),receiver_pool['Product_ID'].isin(donor_pool['Product_ID']).sum(),receiver_pool['Need_Qty'].sum(),sku_imbalance['Total_Donor_Excess'].sum(),sku_imbalance['Max_Matched_Qty_SKU'].sum(),sku_imbalance['Max_Matched_Qty_SKU'].sum()/receiver_pool['Need_Qty'].sum(),sku_imbalance['Covered_GP_Opportunity'].sum(),sku_imbalance['Covered_GP_Opportunity'].sum()/receiver_pool['GP_Opportunity_Exposure'].sum()]})
display(donor_availability_summary); display(sku_imbalance.sort_values(['Total_Receiver_Need','Quantity_Coverage'],ascending=[False,True]))

### C. Action Classification

分配顺序保持透明：Active Stockout优先于Critical；同一SKU内再按GP Opportunity较高者优先。同一SKU的ABC属性相同，因此ABC继续用于风险解释，但不作为同SKU内部的实际分配排序条件。一个Receiver即使获得部分调拨，也可能仍有Remaining Need。

In [ ]:
action_priority=receiver_pool.copy(); action_priority['Status_Rank']=action_priority['Inventory_Status'].map({'Active Stockout':1,'Critical':2}); action_priority=action_priority.sort_values(['Product_ID','Status_Rank','GP_Opportunity_Exposure','Need_Qty','Store_ID'],ascending=[True,True,False,False,True]).copy(); action_priority['Potential_Local_Match']=0; action_priority['Potential_Cross_City_Match']=0
remaining_excess={(int(row.Store_ID),int(row.Product_ID)):int(row.Excess_Qty) for row in donor_pool.itertuples()}
store_city_map=stores.set_index('Store_ID')['Store_City'].to_dict()
for idx,row in action_priority.iterrows():
    need=int(row['Need_Qty']); local_keys=[key for key,qty in remaining_excess.items() if qty>0 and key[1]==int(row['Product_ID']) and store_city_map[key[0]]==row['Store_City'] and key[0]!=int(row['Store_ID'])]
    for key in sorted(local_keys):
        allocated=min(need,remaining_excess[key]); action_priority.at[idx,'Potential_Local_Match']+=allocated; remaining_excess[key]-=allocated; need-=allocated
        if need==0: break
for idx,row in action_priority.iterrows():
    need=int(row['Need_Qty']-action_priority.at[idx,'Potential_Local_Match']); cross_keys=[key for key,qty in remaining_excess.items() if qty>0 and key[1]==int(row['Product_ID']) and store_city_map[key[0]]!=row['Store_City'] and key[0]!=int(row['Store_ID'])]
    for key in sorted(cross_keys):
        allocated=min(need,remaining_excess[key]); action_priority.at[idx,'Potential_Cross_City_Match']+=allocated; remaining_excess[key]-=allocated; need-=allocated
        if need==0: break
action_priority['Matched_Qty']=action_priority['Potential_Local_Match']+action_priority['Potential_Cross_City_Match']; action_priority['Remaining_Need']=action_priority['Need_Qty']-action_priority['Matched_Qty']; action_priority['Recommended_Action']=np.select([action_priority['Potential_Local_Match'].gt(0),action_priority['Potential_Cross_City_Match'].gt(0)],['Same-City Reallocation Candidate','Cross-City Reallocation Candidate'],default='Replenishment Priority')
assert action_priority['Remaining_Need'].ge(0).all(); assert action_priority['Matched_Qty'].sum()==sku_imbalance['Max_Matched_Qty_SKU'].sum()

In [ ]:
action_columns=['Store_ID','Store_Name','Store_City','Product_ID','Product_Name','ABC_Class','Inventory_Status','Stock_On_Hand','Avg_Daily_Demand_90D','Days_Cover','Need_Qty','Potential_Local_Match','Potential_Cross_City_Match','Remaining_Need','Sales_Opportunity_Exposure','GP_Opportunity_Exposure','Recommended_Action']
action_summary=pd.DataFrame({'Metric':['Same-City matched Receiver','Cross-City matched Receiver','Same-City matched units','Cross-City matched units','Receiver with Remaining Need','Remaining Replenishment Need Units','Fully covered Receiver','Inventory Data Exception','Recent-demand Data Investigation Priority'],'Value':[action_priority['Potential_Local_Match'].gt(0).sum(),action_priority['Potential_Cross_City_Match'].gt(0).sum(),action_priority['Potential_Local_Match'].sum(),action_priority['Potential_Cross_City_Match'].sum(),action_priority['Remaining_Need'].gt(0).sum(),action_priority['Remaining_Need'].sum(),action_priority['Remaining_Need'].eq(0).sum(),len(inventory_exceptions),len(data_quality_investigation_priority)]})
display(action_summary)
action_priority_view=action_priority[action_columns].sort_values(['Remaining_Need','GP_Opportunity_Exposure'],ascending=[False,False]).head(20)
display(action_priority_view)

### D. Same-City Constraint

Same-City作为物流可执行性的保守下限。Store_City仍只是简单地理代理，不能替代真实距离、运输成本、Lead Time与配送网络。

In [ ]:
same_city_receiver=receiver_pool.groupby(['Product_ID','Store_City']).agg(Need_Qty=('Need_Qty','sum'),Unit_GP=('Unit_GP','first')); same_city_donor=donor_pool.groupby(['Product_ID','Store_City'])['Excess_Qty'].sum(); same_city_match=same_city_receiver.join(same_city_donor).fillna({'Excess_Qty':0}); same_city_match['Matched_Qty']=np.minimum(same_city_match['Need_Qty'],same_city_match['Excess_Qty']); same_city_match['Covered_GP']=same_city_match['Matched_Qty']*same_city_match['Unit_GP']
same_city_available=set(map(tuple,donor_pool[['Product_ID','Store_City']].drop_duplicates().to_numpy())); same_city_receiver_hits=sum(tuple(values) in same_city_available for values in receiver_pool[['Product_ID','Store_City']].to_numpy())
same_city_summary=pd.DataFrame({'Metric':['Receiver Need','Maximum Matched Units','Unit Need Coverage','Total GP Opportunity Exposure','Covered GP Opportunity','GP Opportunity Coverage','Receiver with Same-City Donor','Receiver Match Rate'],'Value':[receiver_pool['Need_Qty'].sum(),same_city_match['Matched_Qty'].sum(),same_city_match['Matched_Qty'].sum()/receiver_pool['Need_Qty'].sum(),receiver_pool['GP_Opportunity_Exposure'].sum(),same_city_match['Covered_GP'].sum(),same_city_match['Covered_GP'].sum()/receiver_pool['GP_Opportunity_Exposure'].sum(),same_city_receiver_hits,same_city_receiver_hits/len(receiver_pool)]}); display(same_city_summary)

## 06. Scenario Validation & Business Recommendations

本章只对前述决策进行压力测试并总结业务建议，不新增分析方法，也不寻找最好看的参数。

### A. Policy Scenario

Receiver Target固定14D，仅比较Conservative 60D、Base 45D与Aggressive 30D Donor Reserve。

In [ ]:
def calculate_reallocation_scenario(demand_window_days=90,receiver_target_days=14,donor_reserve_days=45):
    window_start=max_sales_date-pd.Timedelta(days=demand_window_days-1)
    demand=sales.loc[sales['Date'].between(window_start,max_sales_date)].groupby(['Store_ID','Product_ID'])['Units'].sum().rename('Window_Units')
    frame=inventory.merge(products,on='Product_ID',validate='many_to_one').merge(stores,on='Store_ID',validate='many_to_one').join(demand,on=['Store_ID','Product_ID']); frame['Window_Units']=frame['Window_Units'].fillna(0); frame['ADD']=frame['Window_Units']/demand_window_days; frame['Days_Cover']=np.where(frame['Window_Units'].gt(0),frame['Stock_On_Hand']/frame['ADD'],np.nan); frame['Unit_GP']=frame['Product_Price']-frame['Product_Cost']
    active=frame.loc[frame['Stock_On_Hand'].eq(0)&frame['Window_Units'].gt(0)]; critical=frame.loc[frame['Stock_On_Hand'].gt(0)&frame['Days_Cover'].gt(0)&frame['Days_Cover'].le(7)]; receivers=pd.concat([active,critical]).copy(); receivers['Need_Qty']=np.maximum(np.ceil(receivers['ADD']*receiver_target_days-receivers['Stock_On_Hand']),0).astype(int); receivers['GP_Opportunity']=receivers['Need_Qty']*receivers['Unit_GP']
    donors=frame.loc[frame['Window_Units'].gt(0)&frame['Days_Cover'].gt(donor_reserve_days)].copy(); donors['Excess_Qty']=np.maximum(np.floor(donors['Stock_On_Hand']-donors['ADD']*donor_reserve_days),0).astype(int); donors=donors.loc[donors['Excess_Qty'].gt(0)]
    receiver_sku=receivers.groupby('Product_ID').agg(Need_Qty=('Need_Qty','sum'),Unit_GP=('Unit_GP','first')); donor_sku=donors.groupby('Product_ID')['Excess_Qty'].sum(); match=receiver_sku.join(donor_sku).fillna({'Excess_Qty':0}); match['Matched_Qty']=np.minimum(match['Need_Qty'],match['Excess_Qty']); match['Covered_GP']=match['Matched_Qty']*match['Unit_GP']
    return {'Active_Stockout':len(active),'Critical':len(critical),'Receiver_Count':len(receivers),'Receiver_Need':int(receivers['Need_Qty'].sum()),'Effective_Donor_Count':len(donors),'Donor_Excess':int(donors['Excess_Qty'].sum()),'Max_Matched_Units':int(match['Matched_Qty'].sum()),'Unit_Coverage':match['Matched_Qty'].sum()/receivers['Need_Qty'].sum(),'Total_GP_Opportunity':receivers['GP_Opportunity'].sum(),'Covered_GP_Opportunity':match['Covered_GP'].sum(),'GP_Coverage':match['Covered_GP'].sum()/receivers['GP_Opportunity'].sum()},receivers,donors,match

In [ ]:
policy_definitions=[('Conservative',60),('Base',45),('Aggressive',30)]; policy_rows=[]
for scenario_name,reserve_days in policy_definitions:
    result,_,_,_=calculate_reallocation_scenario(90,14,reserve_days); policy_rows.append({'Scenario':scenario_name,'Demand_Window_Days':90,'Receiver_Target_Days':14,'Donor_Reserve_Days':reserve_days,**result})
policy_sensitivity=pd.DataFrame(policy_rows); display(policy_sensitivity)
fig,ax=plt.subplots(figsize=(8,4.5)); ax.bar(policy_sensitivity['Scenario'],policy_sensitivity['Unit_Coverage'],color=['#6C757D','#2F6690','#E07A5F'],label='Unit Coverage'); ax.plot(policy_sensitivity['Scenario'],policy_sensitivity['GP_Coverage'],color='#333333',marker='o',label='GP Coverage'); ax.set_title('Reallocation Scenario Coverage'); ax.set_ylabel('Coverage'); ax.yaxis.set_major_formatter(lambda x,pos:f'{x:.0%}'); ax.legend(); fig.tight_layout(); plt.show()

### B. Demand Window Sensitivity

Target固定14D、Reserve固定45D，比较30D、60D与90D历史需求窗口，只判断核心定性结论是否反转。

In [ ]:
demand_window_rows=[]
for window_days in [30,60,90]:
    result,_,_,_=calculate_reallocation_scenario(window_days,14,45); demand_window_rows.append({'Demand_Window':f'{window_days}D',**result})
demand_window_sensitivity=pd.DataFrame(demand_window_rows); display(demand_window_sensitivity[['Demand_Window','Active_Stockout','Critical','Receiver_Count','Receiver_Need','Effective_Donor_Count','Donor_Excess','Max_Matched_Units','Unit_Coverage','Total_GP_Opportunity','Covered_GP_Opportunity','GP_Coverage']])

**Sensitivity interpretation：**不同需求窗口不需要得到相同数字。只要各窗口均显示同SKU的紧缺与Excess同时存在、调拨可覆盖部分但不能覆盖全部Need、高GP商品存在可调拨机会，核心定性结论即保持稳定。90D作为Base用于平滑短期波动；30D和60D保留为透明的敏感性边界。

### C. Final Business Recommendations

最终建议压缩为经营表现、商品优先级、库存风险和库存决策四个层次。所有Scenario、Opportunity、Candidate与Priority均为决策支持，不是实际执行或优化收益。

In [ ]:
base_policy=policy_sensitivity.loc[policy_sensitivity['Scenario'].eq('Base')].iloc[0]
health_lookup=inventory_health_summary.set_index('Inventory_Status'); abc_lookup=abc_summary.set_index('ABC_Class')
final_business_findings=pd.DataFrame([
 {'Business_Conclusion':'1. Business Performance','Finding':f"Total Sales {total_sales:,.2f}，Gross Profit {total_gross_profit:,.2f}，Gross Margin {gross_margin:.2%}；{target_month:%Y-%m}为largest observed monthly decline，Sales MoM {target_sales/previous_sales-1:.2%}，变化{total_delta_sales:,.2f}"},
 {'Business_Conclusion':'2. Product Priority','Finding':f"A类商品{int(abc_lookup.loc['A','Product_Count'])}个，贡献{abc_lookup.loc['A','GP_Share']:.2%}历史Gross Profit，应作为后续库存风险的高价值优先层"},
 {'Business_Conclusion':'3. Inventory Risk','Finding':f"Active Stockout {int(health_lookup.loc['Active Stockout','Store_SKU_Count'])}个，Critical {int(health_lookup.loc['Critical','Store_SKU_Count'])}个；A类Active/Critical Need 4,815件、GP Opportunity 16,972.00；Dormant {int(health_lookup.loc['Dormant / No Recent Sales','Store_SKU_Count'])}个"},
 {'Business_Conclusion':'4. Inventory Decision','Finding':f"Base Reallocation覆盖{base_policy['Unit_Coverage']:.2%} Need与{base_policy['GP_Coverage']:.2%} GP Opportunity；Same-City仅匹配{int(action_priority['Potential_Local_Match'].sum()):,}件；仍有{int(action_priority['Remaining_Need'].sum()):,}件进入补货优先级"}
]); display(final_business_findings)

### Limitations

项目限制决定了本项目是库存配置决策分析，而不是Forecasting、Inventory Optimization或Supply Chain Optimization。

1. inventory只有单时点快照，不能形成库存趋势；
2. 历史销量不等于无约束真实需求，缺货可能压低观察销量；
3. 没有Lead Time；
4. 没有采购订单或在途库存；
5. 没有企业Safety Stock政策；
6. 没有门店最低展示库存；
7. 没有门店距离；
8. 没有运输成本；
9. 没有调拨时间或服务水平约束；
10. Dormant不代表永久滞销；
11. Opportunity Exposure/Coverage不是实际损失、收益或挽回利润；
12. Reallocation Scenario不是生产级优化模型或最优调拨方案；
13. 157个缺失inventory组合不能视为零库存。

### Final Power BI Output Interface

This section freezes the reproducible Python-to-Power-BI interface without adding analysis. Seven outputs keep distinct grains. Products and stores are shared dimensions; fact tables do not relate directly; `scenario_summary` stays disconnected.

In [ ]:
# Build final sales, product, store, inventory, action, and exception interfaces.
sales_fact_output=sales_data[['Sale_ID','Date','Store_ID','Product_ID','Units','Sales','COGS','Gross_Profit']].copy()
products_output=(products[['Product_ID','Product_Name','Product_Category','Product_Cost','Product_Price']].merge(product_abc[['Product_ID','Sales','Gross_Profit','GP_Share','Cumulative_GP_Share','ABC_Class']],on='Product_ID',validate='one_to_one').rename(columns={'Sales':'Historical_Sales','Gross_Profit':'Historical_Gross_Profit'}))
stores_output=stores[['Store_ID','Store_Name','Store_City','Store_Location','Store_Open_Date']].copy()
receiver_keys=pd.MultiIndex.from_frame(receiver_pool[['Store_ID','Product_ID']]); donor_excess=donor_pool.set_index(['Store_ID','Product_ID'])['Excess_Qty']
inventory_fact_output=inventory_analysis[['Store_ID','Product_ID','Stock_On_Hand','Inventory_Cost','Inventory_Retail_Value','Recent_90D_Units','Avg_Daily_Demand_90D','Days_Cover','Inventory_Status','Analytical_Priority']].copy(); inv_keys=pd.MultiIndex.from_frame(inventory_fact_output[['Store_ID','Product_ID']])
inventory_fact_output['Need_Qty_Base']=np.where(inv_keys.isin(receiver_keys),inventory_analysis['Analytical_Need_14D'],0).astype(int)
inventory_fact_output['Sales_Opportunity_Exposure_Base']=inventory_fact_output['Need_Qty_Base']*inventory_analysis['Product_Price'].to_numpy(); inventory_fact_output['GP_Opportunity_Exposure_Base']=inventory_fact_output['Need_Qty_Base']*inventory_analysis['Unit_GP'].to_numpy(); inventory_fact_output['Donor_Excess_Base']=inv_keys.map(donor_excess).fillna(0).astype(int); inventory_fact_output=inventory_fact_output.rename(columns={'Days_Cover':'Days_Cover_90D'})
inventory_actions_output=action_priority[['Store_ID','Product_ID','Inventory_Status','Analytical_Priority','Stock_On_Hand','Avg_Daily_Demand_90D','Days_Cover','Need_Qty','Potential_Local_Match','Potential_Cross_City_Match','Matched_Qty','Remaining_Need','Sales_Opportunity_Exposure','GP_Opportunity_Exposure','Unit_GP','Recommended_Action']].copy().rename(columns={'Days_Cover':'Days_Cover_90D','Potential_Local_Match':'Same_City_Matched_Qty','Potential_Cross_City_Match':'Cross_City_Matched_Qty','Matched_Qty':'Total_Matched_Qty','Remaining_Need':'Remaining_Need_Qty'})
inventory_actions_output['Covered_GP_Opportunity']=inventory_actions_output['Total_Matched_Qty']*inventory_actions_output['Unit_GP']; inventory_actions_output=inventory_actions_output.drop(columns='Unit_GP')
inventory_exceptions_output=inventory_exceptions[['Store_ID','Product_ID','Historical_Units','Recent_90D_Units']].copy().astype({'Historical_Units':'int64','Recent_90D_Units':'int64'})
inventory_exceptions_output['Has_Historical_Sales']=inventory_exceptions_output['Historical_Units'].gt(0); inventory_exceptions_output['Has_Recent_90D_Sales']=inventory_exceptions_output['Recent_90D_Units'].gt(0); inventory_exceptions_output['Exception_Type']='Inventory Record Missing / Inventory Unknown'

In [ ]:
# Build scenario-level outputs; no Store-SKU scenario detail is exported.
scenario_rows=[]
for scenario_name,reserve_days in [('Conservative',60),('Base',45),('Aggressive',30)]:
    result,r,d,_=calculate_reallocation_scenario(90,14,reserve_days); scenario_rows.append({'Scenario_Group':'Policy Scenario','Scenario_Name':scenario_name,'Demand_Window_Days':90,'Receiver_Target_Days':14,'Donor_Reserve_Days':reserve_days,'Geographic_Constraint':'None',**result,'Receiver_Match_Rate':r['Product_ID'].isin(d['Product_ID']).mean()})
for window_days in [30,60,90]:
    result,r,d,_=calculate_reallocation_scenario(window_days,14,45); scenario_rows.append({'Scenario_Group':'Demand Window Sensitivity','Scenario_Name':f'{window_days}D','Demand_Window_Days':window_days,'Receiver_Target_Days':14,'Donor_Reserve_Days':45,'Geographic_Constraint':'None',**result,'Receiver_Match_Rate':r['Product_ID'].isin(d['Product_ID']).mean()})
scenario_rows.append({'Scenario_Group':'Geographic Constraint','Scenario_Name':'Same-City Base','Demand_Window_Days':90,'Receiver_Target_Days':14,'Donor_Reserve_Days':45,'Geographic_Constraint':'Same City','Active_Stockout':int((receiver_pool['Inventory_Status']=='Active Stockout').sum()),'Critical':int((receiver_pool['Inventory_Status']=='Critical').sum()),'Receiver_Count':len(receiver_pool),'Receiver_Need':int(receiver_pool['Need_Qty'].sum()),'Effective_Donor_Count':len(donor_pool),'Donor_Excess':int(donor_pool['Excess_Qty'].sum()),'Max_Matched_Units':int(same_city_match['Matched_Qty'].sum()),'Unit_Coverage':same_city_match['Matched_Qty'].sum()/receiver_pool['Need_Qty'].sum(),'Total_GP_Opportunity':receiver_pool['GP_Opportunity_Exposure'].sum(),'Covered_GP_Opportunity':same_city_match['Covered_GP'].sum(),'GP_Coverage':same_city_match['Covered_GP'].sum()/receiver_pool['GP_Opportunity_Exposure'].sum(),'Receiver_Match_Rate':same_city_receiver_hits/len(receiver_pool)})
scenario_summary_output=pd.DataFrame(scenario_rows)[['Scenario_Group','Scenario_Name','Demand_Window_Days','Receiver_Target_Days','Donor_Reserve_Days','Geographic_Constraint','Active_Stockout','Critical','Receiver_Count','Receiver_Need','Effective_Donor_Count','Donor_Excess','Max_Matched_Units','Unit_Coverage','Total_GP_Opportunity','Covered_GP_Opportunity','GP_Coverage','Receiver_Match_Rate']]

In [ ]:
# Schema audit runs before any CSV is written.
final_output_tables={'sales_fact.csv':sales_fact_output,'products.csv':products_output,'stores.csv':stores_output,'inventory_fact.csv':inventory_fact_output,'inventory_actions.csv':inventory_actions_output,'scenario_summary.csv':scenario_summary_output,'inventory_exceptions.csv':inventory_exceptions_output}
schema_specs={'sales_fact.csv':('One Sale_ID sales record','Sale_ID','products 1:*; stores 1:*; DateTable 1:*'),'products.csv':('One Product_ID','Product_ID','Shared product dimension'),'stores.csv':('One Store_ID','Store_ID','Shared store dimension'),'inventory_fact.csv':('One recorded Store_ID + Product_ID inventory combination','Store_ID + Product_ID','products 1:*; stores 1:*'),'inventory_actions.csv':('One Base Receiver Store_ID + Product_ID','Store_ID + Product_ID','products 1:*; stores 1:*'),'scenario_summary.csv':('One scenario','Scenario_Group + Scenario_Name','Disconnected scenario table'),'inventory_exceptions.csv':('One missing Store_ID + Product_ID combination','Store_ID + Product_ID','products 1:*; stores 1:*')}
schema_audit=pd.DataFrame([{'Table':n,'Grain':schema_specs[n][0],'Rows':len(f),'Key':schema_specs[n][1],'Fields':', '.join(f.columns),'Data_Types':', '.join(f'{c}: {t}' for c,t in f.dtypes.items()),'Relationships':schema_specs[n][2],'Duplicate_Field_Names':f.columns.duplicated().any(),'Many_to_Many_Risk':False} for n,f in final_output_tables.items()]); display(schema_audit)
assert sales_fact_output['Sale_ID'].is_unique and products_output['Product_ID'].is_unique and stores_output['Store_ID'].is_unique
for f in [inventory_fact_output,inventory_actions_output,inventory_exceptions_output]: assert not f.duplicated(['Store_ID','Product_ID']).any()
assert not scenario_summary_output.duplicated(['Scenario_Group','Scenario_Name']).any() and not schema_audit['Duplicate_Field_Names'].any()

In [ ]:
# Write with relative paths, then re-read and validate the actual CSV interface.
OUTPUT_DIR=Path('output'); OUTPUT_DIR.mkdir(exist_ok=True)
for file_name,frame in final_output_tables.items(): frame.to_csv(OUTPUT_DIR/file_name,index=False,date_format='%Y-%m-%d')
reloaded={name:pd.read_csv(OUTPUT_DIR/name) for name in final_output_tables}
status_expected={'Active Stockout':77,'Critical':289,'Low Stock':267,'Healthy':661,'Overstock':206,'Dormant / No Recent Sales':93}
checks={'Sales fact rows':len(reloaded['sales_fact.csv'])==829262,'Sale_ID unique':reloaded['sales_fact.csv']['Sale_ID'].is_unique,'Total Sales':np.isclose(reloaded['sales_fact.csv']['Sales'].sum(),14444572.35),'Total Gross Profit':np.isclose(reloaded['sales_fact.csv']['Gross_Profit'].sum(),4014029.00),'Products rows':len(reloaded['products.csv'])==35,'ABC counts':reloaded['products.csv']['ABC_Class'].value_counts().to_dict()=={'A':15,'C':11,'B':9},'Stores rows':len(reloaded['stores.csv'])==50,'Inventory fact rows':len(reloaded['inventory_fact.csv'])==1593,'Inventory key unique':not reloaded['inventory_fact.csv'].duplicated(['Store_ID','Product_ID']).any(),'Inventory status counts':reloaded['inventory_fact.csv']['Inventory_Status'].value_counts().to_dict()==status_expected,'Action rows':len(reloaded['inventory_actions.csv'])==366,'Action Need':reloaded['inventory_actions.csv']['Need_Qty'].sum()==7622,'Action Matched':reloaded['inventory_actions.csv']['Total_Matched_Qty'].sum()==1426,'Action Remaining':reloaded['inventory_actions.csv']['Remaining_Need_Qty'].sum()==6196,'Same-City Matched':reloaded['inventory_actions.csv']['Same_City_Matched_Qty'].sum()==342,'Exception rows':len(reloaded['inventory_exceptions.csv'])==157,'Exception historical sales':reloaded['inventory_exceptions.csv']['Has_Historical_Sales'].sum()==41,'Exception recent demand':reloaded['inventory_exceptions.csv']['Has_Recent_90D_Sales'].sum()==23,'Scenario rows':len(reloaded['scenario_summary.csv'])==7,'Base matched':int(reloaded['scenario_summary.csv'].query("Scenario_Group == 'Policy Scenario' and Scenario_Name == 'Base'")['Max_Matched_Units'].iloc[0])==1426,'Same-City scenario':int(reloaded['scenario_summary.csv'].query("Scenario_Name == 'Same-City Base'")['Max_Matched_Units'].iloc[0])==342}
output_validation=pd.DataFrame({'Check':checks.keys(),'Passed':checks.values()}); display(output_validation); assert output_validation['Passed'].all(),'Final CSV interface validation failed.'
print('Final Power BI interface written and reloaded successfully:', ', '.join(final_output_tables))